# 🎬 مولّد فيديوهات الأطفال 3D — نسخة Kaggle (مجانية، تشتغل في الخلفية)
**الفرق عن Colab:** تضغط **Save Version** مرة واحدة ثم **تغلق الصفحة وتذهب** — Kaggle يشغّل كل شيء على سيرفراته (ساعات إن لزم)، وتجد `final.mp4` جاهزاً في تبويب **Output** عند الانتهاء.

**الإعداد (مرة واحدة):**
1. حساب Kaggle مجاني، وتوثيق رقم الهاتف (مطلوب لتفعيل الـGPU والإنترنت).
2. من القائمة اليمنى: **Session options** ← **Accelerator = GPU (P100 أو T4)** و **Internet = On**.
3. عدّل قائمة `SCENES` في الخلية 5 (بالعربية أو الإنجليزية، مشهد في كل سطر).
4. **Save Version** ← **Save & Run All (Commit)** ← أغلق الصفحة.
5. بعد الانتهاء: افتح الدفتر ← تبويب **Output** ← حمّل `final.mp4`.

الحصة المجانية للـGPU حوالي 30 ساعة أسبوعياً (تتغير). لم يُجرَّب هذا الدفتر على Kaggle بعد: أرسل لي أي خطأ يظهر في السجل.

In [ ]:
# 1) التثبيت
!pip -q install -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg ftfy deep-translator

In [ ]:
# 2) الإعدادات المشتركة
QUALITY = "fast"   # "fast" سريع | "balanced" متوازن | "high" أعلى جودة (أبطأ بكثير)
PRESETS = {"fast": (33, 20), "balanced": (49, 30), "high": (81, 50)}   # (عدد الإطارات، الخطوات)
NUM_FRAMES, STEPS = PRESETS[QUALITY]
FPS = 16
WIDTH, HEIGHT = 480, 832          # عمودي 9:16 (للأفقي بدّل القيمتين)
UPSCALE_TO = (720, 1280)          # None لإلغاء التكبير
SHARPEN = True                    # زيادة الحدّة
INTERPOLATE_30FPS = False         # True = حركة أنعم (30fps) لكن أبطأ
USE_DRIVE = False                 # (Kaggle) لا حاجة لـ Drive
FP16 = True                       # أسرع على T4 (يرجع تلقائياً إلى bf16 إذا ظهرت صورة سوداء)
NEGATIVE = "Bright tones, overexposed, static, blurred details, subtitles, worst quality, low quality, JPEG artifacts, deformed, extra limbs, ugly, watermark, text"
STYLE = "3D animated children's movie style, adorable big-eyed cartoon characters, vibrant colors, soft cinematic lighting, smooth motion, ultra detailed, sharp focus"

In [ ]:
# 3) تحميل الموديل (مرة واحدة) — بدون مشفّر النص لتفادي انهيار الذاكرة (RAM)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, shutil, subprocess, gc, time
import numpy as np
from diffusers import AutoencoderKLWan, WanPipeline, WanVideoToVideoPipeline
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video, load_video
from transformers import UMT5EncoderModel

WORK = "/kaggle/working"
if False:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK = "/content/drive/MyDrive/kids_video"
        os.makedirs(WORK, exist_ok=True)
    except Exception as e:
        print("تعذّر ربط Drive، سنعمل محلياً:", e)
TE_CACHE = f"{WORK}/umt5_bf16"

model_id = "Wan-AI/Wan2.1-T2V-1.3B-Diffusers"
dtype = torch.float16 if FP16 else torch.bfloat16
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(model_id, vae=vae, text_encoder=None, tokenizer=None, torch_dtype=dtype)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()
try:
    v2v = WanVideoToVideoPipeline.from_pipe(pipe)     # يشارك نفس الأجزاء (يوفّر الذاكرة)
except Exception:
    v2v = WanVideoToVideoPipeline(**pipe.components)

def encode_prompts(prompts):
    """يحمّل مشفّر النص مباشرة على الـ GPU (بلا استهلاك RAM)، يحسب التضمينات، ثم يحرّره."""
    try: pipe.maybe_free_model_hooks()
    except Exception: pass
    gc.collect(); torch.cuda.empty_cache()
    cached = os.path.isdir(TE_CACHE) and os.path.exists(f"{TE_CACHE}/config.json")
    if cached:
        te = UMT5EncoderModel.from_pretrained(TE_CACHE, torch_dtype=torch.bfloat16, device_map={"": 0})
    else:
        te = UMT5EncoderModel.from_pretrained(model_id, subfolder="text_encoder", torch_dtype=torch.bfloat16, device_map={"": 0})
        if False:
            try:
                if shutil.disk_usage(WORK).free > 13e9:
                    te.save_pretrained(TE_CACHE); print("حُفظ مشفّر النص على Drive (لن يُحمَّل من الإنترنت مجدداً).")
                else: print("مساحة Drive لا تكفي لحفظ المشفّر؛ تجاوزنا الحفظ.")
            except Exception as e: print("تعذّر الحفظ على Drive:", e)
    tp = WanPipeline.from_pretrained(model_id, transformer=None, vae=None, text_encoder=te, torch_dtype=torch.bfloat16)
    out = []
    with torch.no_grad():
        for p in prompts:
            pe, ne = tp.encode_prompt(prompt=p, negative_prompt=NEGATIVE, do_classifier_free_guidance=True,
                                      max_sequence_length=512, device="cuda")
            out.append((pe.cpu(), ne.cpu()))
    del tp, te
    gc.collect(); torch.cuda.empty_cache()
    return out

def to_en(t):
    """يترجم الوصف العربي إلى الإنجليزية تلقائياً."""
    if not any("\u0600" <= c <= "\u06ff" for c in t): return t
    try:
        from deep_translator import GoogleTranslator
        return GoogleTranslator(source="auto", target="en").translate(t)
    except Exception:
        print("تعذّرت الترجمة، سأستعمل النص كما هو"); return t

def bad(frames):
    a = np.asarray(frames, dtype=np.float32)
    return (not np.isfinite(a).all()) or a.std() < 0.02

def run_pipe(fn, pe, ne, **kw):
    """يشغّل التوليد؛ إن كان الناتج أسود/تالفاً في fp16 يبدّل إلى bf16 ويعيد المحاولة."""
    for _ in range(2):
        d = pipe.transformer.dtype
        frames = fn(prompt_embeds=pe.to("cuda", d), negative_prompt_embeds=ne.to("cuda", d), **kw).frames[0]
        if not bad(frames): return frames
        if d == torch.float16:
            print("fp16 أعطى ناتجاً تالفاً → التبديل إلى bf16 (أبطأ لكنه أدقّ)")
            pipe.transformer.to(torch.bfloat16)
        else: break
    return frames
print("جاهز | الحفظ في:", WORK)

In [ ]:
import shutil, imageio_ffmpeg
FFMPEG = shutil.which("ffmpeg") or imageio_ffmpeg.get_ffmpeg_exe()
# 4) دالة الدمج والتحسين والتحميل
def finalize(clip_dir, out="final.mp4", audio_src=None, names=None):
    names = names or sorted(n for n in os.listdir(clip_dir) if n.endswith(".mp4") and n != "merged.mp4")
    with open(f"{clip_dir}/list.txt", "w") as f:
        for n in names: f.write(f"file '{n}'\n")
    vf = []
    if UPSCALE_TO: vf.append(f"scale={UPSCALE_TO[0]}:{UPSCALE_TO[1]}:flags=lanczos")
    if SHARPEN: vf.append("unsharp=5:5:0.8:5:5:0.0")
    if INTERPOLATE_30FPS: vf.append("minterpolate=fps=30:mi_mode=mci:mc_mode=aobmc:vsbmc=1")
    vf.append("format=yuv420p")
    merged = f"{clip_dir}/merged.mp4"
    subprocess.run([FFMPEG,"-y","-loglevel","error","-f","concat","-safe","0","-i",f"{clip_dir}/list.txt",
                    "-vf",",".join(vf),"-c:v","libx264","-crf","16","-preset","medium",merged], check=True)
    if audio_src:
        subprocess.run([FFMPEG,"-y","-loglevel","error","-i",merged,"-i",audio_src,"-map","0:v","-map","1:a?",
                        "-c:v","copy","-c:a","aac","-shortest",out], check=True)
    else:
        shutil.copy(merged, out)
    print("✔ الملف النهائي:", os.path.abspath(out))
    return out
print("تمام")

## 5) نص ← فيديو
عدّل `SCENES` ثم احفظ النسخة (Save Version). المقاطع تُحفظ تباعاً، والناتج النهائي `final.mp4` في Output.

In [ ]:
import os, shutil, torch, gc, subprocess, time
if "pipe" not in globals(): raise RuntimeError("شغّل الخلايا 2 و3 و4 أولاً (الجلسة اتعاودت)")
CLEAR = False      # True = ابدأ من الصفر وامسح المقاطع المحفوظة
SCENES = [
    "طفل صغير لطيف بعينين زرقاوين يركب نمراً ودوداً على طريق غابة مشمسة بين الزهور والجبال",
    "a cute baby riding a friendly baby elephant through a colorful flower meadow, a wooden cottage in the background",
    "a cute baby riding a happy panda across a wooden bridge near a waterfall, pink blossom trees",
    "a cute baby riding a playful dolphin in sparkling blue water at sunset, splashing waves",
]
CLIPS = f"{WORK}/clips"
if CLEAR: shutil.rmtree(CLIPS, ignore_errors=True)
os.makedirs(CLIPS, exist_ok=True)
fname = lambda i: f"clip_{i:02d}.mp4"
todo = [(i, s) for i, s in enumerate(SCENES, 1) if not os.path.exists(f"{CLIPS}/{fname(i)}")]
print(f"المقاطع الجاهزة: {len(SCENES) - len(todo)} | المتبقية: {len(todo)}")
if todo:
    print("أحسب تضمين النصوص (قد يستغرق وقتاً في أول مرة) ...")
    embs = encode_prompts([f"{to_en(s)}. {STYLE}" for _, s in todo])
    for (i, _), (pe, ne) in zip(todo, embs):
        t0 = time.time()
        print(f"المقطع {i}/{len(SCENES)} ...")
        frames = run_pipe(pipe, pe, ne, height=HEIGHT, width=WIDTH, num_frames=NUM_FRAMES,
                          num_inference_steps=STEPS, guidance_scale=5.0)
        export_to_video(frames, f"{CLIPS}/{fname(i)}", fps=FPS)
        torch.cuda.empty_cache()
        print(f"  ✔ تم في {(time.time()-t0)/60:.1f} دقيقة")
finalize(CLIPS, out="/kaggle/working/final.mp4", names=[fname(i) for i in range(1, len(SCENES) + 1)])